# IZQAR — IBM AML HI-Small laundering detector
Add Input: `ealtman2019/ibm-transactions-for-anti-money-laundering-aml`. This notebook reads only `HI-Small_Trans.csv`, even though Kaggle mounts the dataset suite. It trains a laundering detector, not a PaySim or IEEE fraud model. Dataset license and provenance: [IBM AML-Data](https://github.com/IBM/AML-Data).

The first run is **validation-only**. Review it before changing `RUN_FINAL` to `True` for one final test evaluation. The source CSV stays in Kaggle. Model scores are uncalibrated and do not establish banking performance.


In [ ]:
from pathlib import Path
import subprocess, sys, json
root = Path('/kaggle/working/izqar_ibm_aml')
if root.exists():
    raise FileExistsError('Use a fresh session; refusing a stale code checkout')
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'feat/ml-pipelines-and-startup',
                'https://github.com/Night011/izqar.git', str(root)], check=True)
backend = root / 'backend'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
                '-r', str(backend / 'requirements-ml.txt'), 'scikit-learn==1.9.1'], check=True)
candidates = [p for p in Path('/kaggle/input').rglob('HI-Small_Trans.csv')
              if 'ibm-transactions-for-anti-money-laundering-aml' in str(p)]
if len(candidates) != 1:
    raise FileNotFoundError(f'Expected one IBM AML HI-Small_Trans.csv via Add Input; found {len(candidates)}')
data = candidates[0]
print('Input:', data, 'bytes:', data.stat().st_size)
print('Code commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=root, text=True).strip())


In [ ]:
experiment = Path('/kaggle/working/ibm_aml_hi_small_validation_v1.joblib')
subprocess.run([sys.executable, str(backend / 'scripts/train_ibm_aml.py'),
                '--data', str(data), '--output', str(experiment),
                '--mode', 'experiment', '--estimators', '180',
                '--max-train-rows', '500000', '--max-false-alerts-per-10000', '100'],
               cwd=backend, check=True)
validation = json.loads(experiment.with_suffix('.metrics.json').read_text())
assert validation['test_metrics'] is None and not experiment.exists()
print('Validation report:', experiment.with_suffix('.metrics.json'))


In [ ]:
RUN_FINAL = False  # Enable only after reviewing validation and freezing settings.
if RUN_FINAL:
    final = Path('/kaggle/working/ibm_aml_hi_small_final_v1.joblib')
    subprocess.run([sys.executable, str(backend / 'scripts/train_ibm_aml.py'),
                    '--data', str(data), '--output', str(final), '--mode', 'final',
                    '--full-train', '--estimators', '180',
                    '--max-false-alerts-per-10000', '100'], cwd=backend, check=True)
    report = json.loads(final.with_suffix('.metrics.json').read_text())
    assert report['model_kind'] == 'ibm_aml' and report['test_metrics'] is not None
    portable = Path('/kaggle/working/izqar_ibm_aml_hi_small_v1_portable.zip')
    subprocess.run([sys.executable, str(backend / 'scripts/export_portable_model.py'),
                    '--source', str(final),
                    '--output-stem', '/kaggle/working/ibm_aml_hi_small_v1_portable',
                    '--bundle', str(portable)], cwd=backend, check=True)
    print('Download trusted model bundle:', portable)
else:
    print('Final test remains sealed. Review validation first.')
